# Detector class recovery — Colab demo

Recover what each class index of an **undocumented YOLO checkpoint** responds to, using a
diffusion model as a natural-image prior instead of psychedelic activation-maximisation images.

Pipeline: **weight-diff** against the base checkpoint → **prompt search** (gradient-free sweep of a
noun vocabulary through the diffusion model) → **guided sampling** (steer the sampler with the
detector's class-score gradient) → **CLIP naming** → **evaluate** on a testbed with known answers.

Runtime → *Change runtime type* → **T4 GPU** (or better) before running.

In [ ]:
#@title 1. Install  (≈2 min)
import os, sys
REPO = "yolo-class-recovery"
if not os.path.exists(REPO):
    !git clone -q https://github.com/dwtrott/yolo-class-recovery.git
%pip install -q -e ./yolo-class-recovery
import importlib, classrecovery; print("classrecovery", classrecovery.__version__)

In [ ]:
#@title 2. Build the testbed: fine-tune YOLOv8n on 4 non-COCO classes, then strip the labels  (≈5 min on T4)
#@markdown The stripped copy is `testbed/mystery.pt` — that's what the recovery pipeline sees.
#@markdown `testbed/truth.json` keeps the real names for scoring only.
from classrecovery.testbed import build_testbed
EPOCHS = 15  #@param {type:"integer"}
paths = build_testbed("testbed", dataset="african-wildlife.yaml", base="yolov8n.pt", epochs=EPOCHS, imgsz=640)
print(paths)
import json; print("ground truth (don't peek if you want to play fair):", json.load(open("testbed/truth.json"))["names"])

In [ ]:
#@title 3. Weight diff against the base checkpoint  (seconds)
from classrecovery import Detector
from classrecovery.weight_diff import weight_diff
mystery, base = Detector("testbed/mystery.pt"), Detector("yolov8n.pt")
print("names in mystery checkpoint:", mystery.names)
print(weight_diff(mystery, base).summary())

In [ ]:
#@title 4. Prompt search over the built-in vocabulary  (≈4 min on T4 with sd-turbo)
from classrecovery.diffusion import DiffusionPrior
from classrecovery.prompt_search import prompt_search
prior = DiffusionPrior("stabilityai/sd-turbo")
search = prompt_search(mystery, prior, n_per_word=2, progress=lambda d,t,p: print(f"\r{d}/{t} {p:<40}", end=""))
print(); print(search.summary(6))
search.save("runs/search")

In [ ]:
#@title 5. Guided recovery + CLIP naming for every class  (≈1 min / class on T4)
from classrecovery.naming import Namer
from classrecovery.pipeline import recover_class, evaluate
from classrecovery.diffusion import GuidanceConfig
from IPython.display import display
namer = Namer()
gcfg = GuidanceConfig(steps=8, cfg=0.0, strength=0.08, repeats=2, n_images=4, seed=0)
results = {}
for c in range(mystery.nc):
    rec = recover_class(mystery, prior, namer, c, search=search, gcfg=gcfg)
    results[c] = rec
    print(f"class {c}: prompt={rec.prompt_used!r}")
    print("   search:", [w for w,_ in rec.search_words[:5]])
    print("   clip:  ", [w for w,_ in rec.clip_names[:5]])
    print("   final: ", [w for w,_ in rec.combined[:5]])
    for im in rec.guided_images[:2]: display(im.resize((256,256)))
truth = {int(k): v for k, v in json.load(open("testbed/truth.json"))["names"].items()}
ev = evaluate(results, truth); print({k: ev[k] for k in ("top1","top3","top5")}); print(ev["per_class"])

In [ ]:
#@title 6. Launch the GUI  (prints a public gradio.live link; also renders inline)
from classrecovery.app import launch
launch(share=True, debug=False)

### Bring your own mystery model
Upload any Ultralytics `.pt` to Colab, then in the GUI's **Model** tab point at it (pick the base
checkpoint it was probably fine-tuned from, or `(none)`).  Give the **Prompt search** tab a
domain-specific vocabulary (one term per line) and a domain prefix such as
`an aerial photo of a {}` if the original classes suggest overhead imagery.